# HW1 — Time-Series Data Analysis: Exploring EHR Data (PhysioNet 2012)

**Setup:** put `P12_preprocess.py`, `Outcomes-a.txt`, `Outcomes-b.txt` and `set-a.tar.gz` / `set-b.tar.gz`
(or the extracted `set-a/`, `set-b/`) in the same folder as this notebook (on Colab: `/content`).

In [1]:
import tarfile
from pathlib import Path

import numpy as np
import pandas as pd

import P12_preprocess as pp
from P12_preprocess import FEATURE_COLS

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

# Extract the raw data if only the archives are present
for d in (pp.SET_A_DIR, pp.SET_B_DIR):
    if not d.is_dir():
        with tarfile.open(f"{d}.tar.gz") as tar:
            tar.extractall(d.parent, filter="data")

PID = 132539  # example patient for (a)-(e)

## P1(a) Data structure

In [2]:
raw = pd.read_csv(pp.SET_A_DIR / f"{PID}.txt")
print("shape:", raw.shape)
raw.head(10)

shape: (273, 3)


,Time,Parameter,Value
0,00:00,RecordID,132539.00
1,00:00,Age,54.00
2,00:00,Gender,0.00
3,00:00,Height,-1.00
4,00:00,ICUType,4.00
5,00:00,Weight,-1.00
6,00:07,GCS,15.00
7,00:07,HR,73.00
8,00:07,NIDiasABP,65.00
9,00:07,NIMAP,92.33


**One row explained** — row 7 (`00:07, HR, 73`):
- `Time`: time elapsed since ICU admission (`HH:MM`, not clock time) → 7 minutes after admission.
- `Parameter`: the variable measured → heart rate.
- `Value`: the measurement → 73 bpm.

**General descriptors vs. time series** — method (ii): scan every file in Set A.
A parameter is a general descriptor if every patient has it **exactly once, at `00:00`**.

In [3]:
long_a = pd.concat(
    [pd.read_csv(f).assign(pid=int(f.stem)) for f in sorted(pp.SET_A_DIR.glob("*.txt"))],
    ignore_index=True,
)
n_patients = long_a.pid.nunique()

per_patient = long_a.groupby(["Parameter", "pid"]).size()
summary = pd.DataFrame({
    "n_patients": per_patient.groupby(level=0).size(),
    "max_per_patient": per_patient.groupby(level=0).max(),
    "frac_at_00:00": long_a.Time.eq("00:00").groupby(long_a.Parameter).mean().round(4),
})
summary.sort_values("frac_at_00:00", ascending=False)

,n_patients,max_per_patient,frac_at_00:00
Parameter,,,
Age,4000,1,1.0000
Gender,4000,1,1.0000
RecordID,4000,1,1.0000
ICUType,4000,1,1.0000
Height,4000,1,1.0000
Weight,4000,179,0.0310
Cholesterol,305,2,0.0032
Albumin,1615,9,0.0013
ALP,1690,12,0.0010


In [4]:
is_descriptor = (
    summary.n_patients.eq(n_patients) & summary.max_per_patient.eq(1) & summary["frac_at_00:00"].eq(1)
)
print(f"Once-at-admission ({is_descriptor.sum()}):", summary.index[is_descriptor].tolist())
print(f"Repeated ({(~is_descriptor).sum()}):", summary.index[~is_descriptor].tolist())

# Weight is recorded at 00:00 for every patient (admission weight) AND repeatedly afterwards
w = long_a[long_a.Parameter == "Weight"]
print(f"\nWeight: at 00:00 for {w[w.Time == '00:00'].pid.nunique()}/{n_patients} patients, "
      f"{(w.Time != '00:00').sum()} later measurements")

Once-at-admission (5): ['Age', 'Gender', 'Height', 'ICUType', 'RecordID']
Repeated (37): ['ALP', 'ALT', 'AST', 'Albumin', 'BUN', 'Bilirubin', 'Cholesterol', 'Creatinine', 'DiasABP', 'FiO2', 'GCS', 'Glucose', 'HCO3', 'HCT', 'HR', 'K', 'Lactate', 'MAP', 'MechVent', 'Mg', 'NIDiasABP', 'NIMAP', 'NISysABP', 'Na', 'PaCO2', 'PaO2', 'Platelets', 'RespRate', 'SaO2', 'SysABP', 'Temp', 'TroponinI', 'TroponinT', 'Urine', 'WBC', 'Weight', 'pH']

Weight: at 00:00 for 4000/4000 patients, 125165 later measurements


- **General descriptors (6):** `RecordID`, `Age`, `Gender`, `Height`, `ICUType`, `Weight` (admission weight).
- **Time series variables (37):** the 37 repeated parameters above. `Weight` belongs to both groups.

## P1(b) Relative time conversion

- Source: `set-a/132539.txt` → output: `132539_hours.txt`
- Method: `hours = HH + MM / 60` (e.g. `00:07` → 0.1167). All columns are read as text, so `Parameter`/`Value`
  are written back exactly as in the source; every row (incl. `RecordID`) is kept in order.

In [5]:
SRC, DST = pp.SET_A_DIR / f"{PID}.txt", Path(f"{PID}_hours.txt")

df_b = pd.read_csv(SRC, dtype=str)  # read as text so Parameter/Value stay unchanged
hh, mm = df_b.Time.str.split(":", expand=True).astype(int).T.to_numpy()
df_b["Time"] = hh + mm / 60
df_b.to_csv(DST, index=False)

# Check: same rows in the same order, Parameter/Value untouched
src, out = pd.read_csv(SRC, dtype=str), pd.read_csv(DST, dtype=str)
assert len(src) == len(out) and src[["Parameter", "Value"]].equals(out[["Parameter", "Value"]])
print("rows:", len(out))
pd.concat({"source": src, "converted": out}, axis=1).head(10)

rows: 273


source                               converted                   
    Time  Parameter   Value                 Time  Parameter   Value
0  00:00   RecordID  132539                  0.0   RecordID  132539
1  00:00        Age      54                  0.0        Age      54
2  00:00     Gender       0                  0.0     Gender       0
3  00:00     Height      -1                  0.0     Height      -1
4  00:00    ICUType       4                  0.0    ICUType       4
5  00:00     Weight      -1                  0.0     Weight      -1
6  00:07        GCS      15  0.11666666666666667        GCS      15
7  00:07         HR      73  0.11666666666666667         HR      73
8  00:07  NIDiasABP      65  0.11666666666666667  NIDiasABP      65
9  00:07      NIMAP   92.33  0.11666666666666667      NIMAP   92.33

## P1(c) Per-patient time series structure (provided `P12_preprocess.py`)

In [6]:
pivot_a = pp.process_set(pp.SET_A_DIR, pp.OUTCOMES_A)
pivot_b = pp.process_set(pp.SET_B_DIR, pp.OUTCOMES_B)
for name, p in [("Set A", pivot_a), ("Set B", pivot_b)]:
    print(f"{name}: {p.shape}, {p.pid.nunique()} patients")

Set A: (93698, 45), 4000 patients
Set B: (93685, 45), 4000 patients


In [7]:
print("Raw (long format):", raw.shape)
display(raw.head(15))

one = pivot_a[pivot_a.pid == PID]
print("Processed (one row per 2-h bin):", one.shape)
display(one)

Raw (long format): (273, 3)


,Time,Parameter,Value
0,00:00,RecordID,132539.00
1,00:00,Age,54.00
2,00:00,Gender,0.00
3,00:00,Height,-1.00
4,00:00,ICUType,4.00
5,00:00,Weight,-1.00
6,00:07,GCS,15.00
7,00:07,HR,73.00
8,00:07,NIDiasABP,65.00
9,00:07,NIMAP,92.33


Processed (one row per 2-h bin): (24, 45)


Parameter,pid,timestamp,Weight,ALP,ALT,AST,Albumin,BUN,Bilirubin,Cholesterol,Creatinine,DiasABP,FiO2,GCS,Glucose,HCO3,HCT,HR,K,Lactate,MAP,MechVent,Mg,NIDiasABP,NIMAP,NISysABP,Na,PaCO2,PaO2,Platelets,RespRate,SaO2,SysABP,Temp,TroponinI,TroponinT,Urine,WBC,pH,Age,Height,Gender,ICUType,target,seg_entry_cnt
0,132539,2,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,70.0,NaN,NaN,NaN,NaN,NaN,61.666667,90.110,147.0,NaN,NaN,NaN,NaN,18.666667,NaN,NaN,35.35,NaN,NaN,330.0,NaN,NaN,54.0,NaN,0.0,4.0,0,4
1,132539,4,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,33.7,71.0,NaN,NaN,NaN,NaN,NaN,52.000000,74.170,118.5,NaN,NaN,NaN,NaN,19.500000,NaN,NaN,37.80,NaN,NaN,115.0,NaN,NaN,54.0,NaN,0.0,4.0,0,3
2,132539,6,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.5,NaN,NaN,NaN,NaN,NaN,45.000000,66.670,110.0,NaN,NaN,NaN,NaN,18.500000,NaN,NaN,NaN,NaN,NaN,170.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
3,132539,8,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,64.0,NaN,NaN,NaN,NaN,NaN,49.000000,68.330,107.0,NaN,NaN,NaN,NaN,15.000000,NaN,NaN,38.10,NaN,NaN,120.0,NaN,NaN,54.0,NaN,0.0,4.0,0,1
4,132539,10,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.0,NaN,NaN,NaN,NaN,NaN,52.000000,70.665,108.0,NaN,NaN,NaN,NaN,15.500000,NaN,NaN,NaN,NaN,NaN,90.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
5,132539,12,-1.0,NaN,NaN,NaN,NaN,13.0,NaN,NaN,0.8,NaN,NaN,15.0,205.0,26.0,33.5,59.5,4.4,NaN,NaN,NaN,1.5,51.000000,69.000,105.0,137.0,NaN,NaN,221.0,15.000000,NaN,NaN,37.60,NaN,NaN,70.0,11.2,NaN,54.0,NaN,0.0,4.0,0,2
6,132539,14,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.0,NaN,NaN,NaN,NaN,NaN,43.500000,63.000,102.0,NaN,NaN,NaN,NaN,13.500000,NaN,NaN,NaN,NaN,NaN,55.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
7,132539,16,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,66.0,NaN,NaN,NaN,NaN,NaN,49.000000,70.000,112.0,NaN,NaN,NaN,NaN,13.500000,NaN,NaN,36.90,NaN,NaN,120.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
8,132539,18,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.0,NaN,NaN,NaN,NaN,NaN,41.500000,60.000,97.0,NaN,NaN,NaN,NaN,13.500000,NaN,NaN,NaN,NaN,NaN,140.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
9,132539,20,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,68.5,NaN,NaN,NaN,NaN,NaN,55.000000,74.670,114.0,NaN,NaN,NaN,NaN,15.500000,NaN,NaN,37.30,NaN,NaN,80.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2


In [8]:
STATIC = ["Age", "Gender", "Height", "ICUType", "Weight"]
print("Distinct values per static column:", one[STATIC].nunique(dropna=False).to_dict())
print("Timestamps:", one.timestamp.tolist())

Distinct values per static column: {'Age': 1, 'Gender': 1, 'Height': 1, 'ICUType': 1, 'Weight': 1}
Timestamps: [2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40, 42, 44, 46, 48]


1. **Missing value:** a parameter not measured in a bin is `NaN` (a bin gets a row only if something was recorded in it).
2. **Static descriptors:** recorded once at `00:00` (→ first bin), then the first known value is copied to **every row**,
   so each static column is constant per patient (`NaN` in all rows if never recorded, e.g. `Height = -1`).
   This also overwrites the later `Weight` measurements with the first value.
3. **Extra columns:** `target` = `In-hospital_death` (0 = survived, 1 = died), repeated on every row;
   `seg_entry_cnt` = number of distinct raw timestamps merged into that bin (bookkeeping, not a feature).
   **Bin size = 2 h**, right-closed `(t-2, t]` and labelled by its end `t` (`00:00` goes to `t = 2`) → up to 24 rows per patient.

## P1(d) Windowing to the study design (12 ≤ timestamp ≤ 36)

In [9]:
def window(df, start=12, end=36):
    """Keep bins with start <= timestamp <= end."""
    return df[df.timestamp.between(start, end)].reset_index(drop=True)

win_a, win_b = window(pivot_a), window(pivot_b)

print("Before:", one.shape)
display(one)
after = win_a[win_a.pid == PID]
print("After:", after.shape, "timestamps:", after.timestamp.tolist())
display(after)

Before: (24, 45)


Parameter,pid,timestamp,Weight,ALP,ALT,AST,Albumin,BUN,Bilirubin,Cholesterol,Creatinine,DiasABP,FiO2,GCS,Glucose,HCO3,HCT,HR,K,Lactate,MAP,MechVent,Mg,NIDiasABP,NIMAP,NISysABP,Na,PaCO2,PaO2,Platelets,RespRate,SaO2,SysABP,Temp,TroponinI,TroponinT,Urine,WBC,pH,Age,Height,Gender,ICUType,target,seg_entry_cnt
0,132539,2,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,70.0,NaN,NaN,NaN,NaN,NaN,61.666667,90.110,147.0,NaN,NaN,NaN,NaN,18.666667,NaN,NaN,35.35,NaN,NaN,330.0,NaN,NaN,54.0,NaN,0.0,4.0,0,4
1,132539,4,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,33.7,71.0,NaN,NaN,NaN,NaN,NaN,52.000000,74.170,118.5,NaN,NaN,NaN,NaN,19.500000,NaN,NaN,37.80,NaN,NaN,115.0,NaN,NaN,54.0,NaN,0.0,4.0,0,3
2,132539,6,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,73.5,NaN,NaN,NaN,NaN,NaN,45.000000,66.670,110.0,NaN,NaN,NaN,NaN,18.500000,NaN,NaN,NaN,NaN,NaN,170.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
3,132539,8,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,64.0,NaN,NaN,NaN,NaN,NaN,49.000000,68.330,107.0,NaN,NaN,NaN,NaN,15.000000,NaN,NaN,38.10,NaN,NaN,120.0,NaN,NaN,54.0,NaN,0.0,4.0,0,1
4,132539,10,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.0,NaN,NaN,NaN,NaN,NaN,52.000000,70.665,108.0,NaN,NaN,NaN,NaN,15.500000,NaN,NaN,NaN,NaN,NaN,90.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
5,132539,12,-1.0,NaN,NaN,NaN,NaN,13.0,NaN,NaN,0.8,NaN,NaN,15.0,205.0,26.0,33.5,59.5,4.4,NaN,NaN,NaN,1.5,51.000000,69.000,105.0,137.0,NaN,NaN,221.0,15.000000,NaN,NaN,37.60,NaN,NaN,70.0,11.2,NaN,54.0,NaN,0.0,4.0,0,2
6,132539,14,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.0,NaN,NaN,NaN,NaN,NaN,43.500000,63.000,102.0,NaN,NaN,NaN,NaN,13.500000,NaN,NaN,NaN,NaN,NaN,55.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
7,132539,16,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,66.0,NaN,NaN,NaN,NaN,NaN,49.000000,70.000,112.0,NaN,NaN,NaN,NaN,13.500000,NaN,NaN,36.90,NaN,NaN,120.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
8,132539,18,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.0,NaN,NaN,NaN,NaN,NaN,41.500000,60.000,97.0,NaN,NaN,NaN,NaN,13.500000,NaN,NaN,NaN,NaN,NaN,140.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
9,132539,20,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,68.5,NaN,NaN,NaN,NaN,NaN,55.000000,74.670,114.0,NaN,NaN,NaN,NaN,15.500000,NaN,NaN,37.30,NaN,NaN,80.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2


After: (13, 45) timestamps: [12, 14, 16, 18, 20, 22, 24, 26, 28, 30, 32, 34, 36]


Parameter,pid,timestamp,Weight,ALP,ALT,AST,Albumin,BUN,Bilirubin,Cholesterol,Creatinine,DiasABP,FiO2,GCS,Glucose,HCO3,HCT,HR,K,Lactate,MAP,MechVent,Mg,NIDiasABP,NIMAP,NISysABP,Na,PaCO2,PaO2,Platelets,RespRate,SaO2,SysABP,Temp,TroponinI,TroponinT,Urine,WBC,pH,Age,Height,Gender,ICUType,target,seg_entry_cnt
0,132539,12,-1.0,NaN,NaN,NaN,NaN,13.0,NaN,NaN,0.8,NaN,NaN,15.0,205.0,26.0,33.5,59.5,4.4,NaN,NaN,NaN,1.5,51.0,69.00,105.0,137.0,NaN,NaN,221.0,15.0,NaN,NaN,37.6,NaN,NaN,70.0,11.2,NaN,54.0,NaN,0.0,4.0,0,2
1,132539,14,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.0,NaN,NaN,NaN,NaN,NaN,43.5,63.00,102.0,NaN,NaN,NaN,NaN,13.5,NaN,NaN,NaN,NaN,NaN,55.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
2,132539,16,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,66.0,NaN,NaN,NaN,NaN,NaN,49.0,70.00,112.0,NaN,NaN,NaN,NaN,13.5,NaN,NaN,36.9,NaN,NaN,120.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
3,132539,18,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.0,NaN,NaN,NaN,NaN,NaN,41.5,60.00,97.0,NaN,NaN,NaN,NaN,13.5,NaN,NaN,NaN,NaN,NaN,140.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
4,132539,20,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,68.5,NaN,NaN,NaN,NaN,NaN,55.0,74.67,114.0,NaN,NaN,NaN,NaN,15.5,NaN,NaN,37.3,NaN,NaN,80.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
5,132539,22,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.5,NaN,NaN,NaN,NaN,NaN,44.0,63.33,102.0,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,70.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
6,132539,24,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,68.5,NaN,NaN,NaN,NaN,NaN,41.5,62.17,103.5,NaN,NaN,NaN,NaN,19.5,NaN,NaN,37.1,NaN,NaN,60.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
7,132539,26,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.0,NaN,NaN,NaN,NaN,NaN,48.0,67.00,105.0,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,115.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
8,132539,28,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,80.0,NaN,NaN,NaN,NaN,NaN,39.0,60.67,104.0,NaN,NaN,NaN,NaN,18.5,NaN,NaN,37.9,NaN,NaN,200.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
9,132539,30,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.0,NaN,NaN,NaN,NaN,NaN,41.0,63.33,108.0,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,90.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2


- **Why predict at hour 36:** P12 only includes ICU stays of at least 48 h, so the outcome is still ahead of the anchor;
  a risk score at 36 h leaves time to act (escalate care, adjust treatment, talk with the family),
  while a full day of vitals and at least one round of labs are already available.
- **Why start at hour 12, not 0:** the first hours mostly reflect the admission itself (surgery recovery, sedation,
  resuscitation, transfer) rather than the patient's own condition, and data are sparse and irregular then.
  Starting at 12 h gives every patient the same 24-h window of more stable, comparable physiology.
- Note: bins are right-closed, so bin 12 = `(10, 12]` and the window covers hours `(10, 36]`.

## P1(e) Missing value imputation

1. **Sentinels → NaN:** `-1` means "not recorded" for `Weight` and `Gender` (`Height` is already handled by `P12_preprocess.py`).
2. **Within each patient: forward-fill, then back-fill** across the window bins. A measurement stays valid until it
   is re-measured (as clinicians read charts), and every value used comes from inside the 12–36 h window,
   so nothing after the anchor leaks in.
3. **Never measured in the window → Set A statistic:** median for continuous variables (robust to outliers,
   e.g. `pH` up to 735), mode for `Gender`/`ICUType`, and `0` for `MechVent` (it is only ever recorded as 1,
   so "not recorded" means "not ventilated"). Statistics come from Set A only; the test set never informs preprocessing.

In [10]:
SENTINEL_COLS = ["Weight", "Gender"]
CATEGORICAL = ["Gender", "ICUType"]

def clean_sentinels(df):
    """-1 means 'not recorded'."""
    df = df.copy()
    df[SENTINEL_COLS] = df[SENTINEL_COLS].replace(-1, np.nan)
    return df

def fit_fallback(win):
    """Fallback values for never-measured features, learned from the given (training) rows only."""
    clean = clean_sentinels(win)
    fallback = clean[FEATURE_COLS].median()
    fallback[CATEGORICAL] = clean[CATEGORICAL].mode().iloc[0]
    fallback["MechVent"] = 0
    return fallback

def impute(df, fallback):
    """Sentinels -> NaN, per-patient ffill/bfill, then fallback values."""
    df = clean_sentinels(df)
    filled = df.groupby("pid")[FEATURE_COLS].ffill().groupby(df.pid).bfill()
    df[FEATURE_COLS] = filled.fillna(fallback)
    return df

fallback = fit_fallback(win_a)  # Set A (training) only
imp_a, imp_b = impute(win_a, fallback), impute(win_b, fallback)
print("Remaining NaNs - Set A:", imp_a[FEATURE_COLS].isna().sum().sum(),
      "| Set B:", imp_b[FEATURE_COLS].isna().sum().sum())

Remaining NaNs - Set A: 0 | Set B: 0


In [11]:
before, after = win_a[win_a.pid == PID], imp_a[imp_a.pid == PID]
print(f"Patient {PID} - before: {before[FEATURE_COLS].isna().sum().sum()} NaNs")
display(before)
print(f"After: {after[FEATURE_COLS].isna().sum().sum()} NaNs")
display(after)

Patient 132539 - before: 369 NaNs


Parameter,pid,timestamp,Weight,ALP,ALT,AST,Albumin,BUN,Bilirubin,Cholesterol,Creatinine,DiasABP,FiO2,GCS,Glucose,HCO3,HCT,HR,K,Lactate,MAP,MechVent,Mg,NIDiasABP,NIMAP,NISysABP,Na,PaCO2,PaO2,Platelets,RespRate,SaO2,SysABP,Temp,TroponinI,TroponinT,Urine,WBC,pH,Age,Height,Gender,ICUType,target,seg_entry_cnt
0,132539,12,-1.0,NaN,NaN,NaN,NaN,13.0,NaN,NaN,0.8,NaN,NaN,15.0,205.0,26.0,33.5,59.5,4.4,NaN,NaN,NaN,1.5,51.0,69.00,105.0,137.0,NaN,NaN,221.0,15.0,NaN,NaN,37.6,NaN,NaN,70.0,11.2,NaN,54.0,NaN,0.0,4.0,0,2
1,132539,14,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,59.0,NaN,NaN,NaN,NaN,NaN,43.5,63.00,102.0,NaN,NaN,NaN,NaN,13.5,NaN,NaN,NaN,NaN,NaN,55.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
2,132539,16,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,66.0,NaN,NaN,NaN,NaN,NaN,49.0,70.00,112.0,NaN,NaN,NaN,NaN,13.5,NaN,NaN,36.9,NaN,NaN,120.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
3,132539,18,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,63.0,NaN,NaN,NaN,NaN,NaN,41.5,60.00,97.0,NaN,NaN,NaN,NaN,13.5,NaN,NaN,NaN,NaN,NaN,140.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
4,132539,20,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,68.5,NaN,NaN,NaN,NaN,NaN,55.0,74.67,114.0,NaN,NaN,NaN,NaN,15.5,NaN,NaN,37.3,NaN,NaN,80.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
5,132539,22,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.5,NaN,NaN,NaN,NaN,NaN,44.0,63.33,102.0,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,70.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
6,132539,24,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,68.5,NaN,NaN,NaN,NaN,NaN,41.5,62.17,103.5,NaN,NaN,NaN,NaN,19.5,NaN,NaN,37.1,NaN,NaN,60.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
7,132539,26,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,67.0,NaN,NaN,NaN,NaN,NaN,48.0,67.00,105.0,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,115.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
8,132539,28,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15.0,NaN,NaN,NaN,80.0,NaN,NaN,NaN,NaN,NaN,39.0,60.67,104.0,NaN,NaN,NaN,NaN,18.5,NaN,NaN,37.9,NaN,NaN,200.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2
9,132539,30,-1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,80.0,NaN,NaN,NaN,NaN,NaN,41.0,63.33,108.0,NaN,NaN,NaN,NaN,16.0,NaN,NaN,NaN,NaN,NaN,90.0,NaN,NaN,54.0,NaN,0.0,4.0,0,2


After: 0 NaNs


Parameter,pid,timestamp,Weight,ALP,ALT,AST,Albumin,BUN,Bilirubin,Cholesterol,Creatinine,DiasABP,FiO2,GCS,Glucose,HCO3,HCT,HR,K,Lactate,MAP,MechVent,Mg,NIDiasABP,NIMAP,NISysABP,Na,PaCO2,PaO2,Platelets,RespRate,SaO2,SysABP,Temp,TroponinI,TroponinT,Urine,WBC,pH,Age,Height,Gender,ICUType,target,seg_entry_cnt
0,132539,12,78.4,81.0,52.0,74.0,2.8,13.0,1.0,144.0,0.8,58.0,0.5,15.0,205.0,26.0,33.5,59.5,4.4,1.9,78.0,0.0,1.5,51.0,69.00,105.0,137.0,39.0,110.0,221.0,15.0,97.0,117.0,37.6,3.1,0.24,70.0,11.2,7.39,54.0,170.2,0.0,4.0,0,2
1,132539,14,78.4,81.0,52.0,74.0,2.8,13.0,1.0,144.0,0.8,58.0,0.5,15.0,205.0,26.0,33.5,59.0,4.4,1.9,78.0,0.0,1.5,43.5,63.00,102.0,137.0,39.0,110.0,221.0,13.5,97.0,117.0,37.6,3.1,0.24,55.0,11.2,7.39,54.0,170.2,0.0,4.0,0,2
2,132539,16,78.4,81.0,52.0,74.0,2.8,13.0,1.0,144.0,0.8,58.0,0.5,15.0,205.0,26.0,33.5,66.0,4.4,1.9,78.0,0.0,1.5,49.0,70.00,112.0,137.0,39.0,110.0,221.0,13.5,97.0,117.0,36.9,3.1,0.24,120.0,11.2,7.39,54.0,170.2,0.0,4.0,0,2
3,132539,18,78.4,81.0,52.0,74.0,2.8,13.0,1.0,144.0,0.8,58.0,0.5,15.0,205.0,26.0,33.5,63.0,4.4,1.9,78.0,0.0,1.5,41.5,60.00,97.0,137.0,39.0,110.0,221.0,13.5,97.0,117.0,36.9,3.1,0.24,140.0,11.2,7.39,54.0,170.2,0.0,4.0,0,2
4,132539,20,78.4,81.0,52.0,74.0,2.8,13.0,1.0,144.0,0.8,58.0,0.5,15.0,205.0,26.0,33.5,68.5,4.4,1.9,78.0,0.0,1.5,55.0,74.67,114.0,137.0,39.0,110.0,221.0,15.5,97.0,117.0,37.3,3.1,0.24,80.0,11.2,7.39,54.0,170.2,0.0,4.0,0,2
5,132539,22,78.4,81.0,52.0,74.0,2.8,13.0,1.0,144.0,0.8,58.0,0.5,15.0,205.0,26.0,33.5,67.5,4.4,1.9,78.0,0.0,1.5,44.0,63.33,102.0,137.0,39.0,110.0,221.0,16.0,97.0,117.0,37.3,3.1,0.24,70.0,11.2,7.39,54.0,170.2,0.0,4.0,0,2
6,132539,24,78.4,81.0,52.0,74.0,2.8,13.0,1.0,144.0,0.8,58.0,0.5,15.0,205.0,26.0,33.5,68.5,4.4,1.9,78.0,0.0,1.5,41.5,62.17,103.5,137.0,39.0,110.0,221.0,19.5,97.0,117.0,37.1,3.1,0.24,60.0,11.2,7.39,54.0,170.2,0.0,4.0,0,2
7,132539,26,78.4,81.0,52.0,74.0,2.8,13.0,1.0,144.0,0.8,58.0,0.5,15.0,205.0,26.0,33.5,67.0,4.4,1.9,78.0,0.0,1.5,48.0,67.00,105.0,137.0,39.0,110.0,221.0,16.0,97.0,117.0,37.1,3.1,0.24,115.0,11.2,7.39,54.0,170.2,0.0,4.0,0,2
8,132539,28,78.4,81.0,52.0,74.0,2.8,13.0,1.0,144.0,0.8,58.0,0.5,15.0,205.0,26.0,33.5,80.0,4.4,1.9,78.0,0.0,1.5,39.0,60.67,104.0,137.0,39.0,110.0,221.0,18.5,97.0,117.0,37.9,3.1,0.24,200.0,11.2,7.39,54.0,170.2,0.0,4.0,0,2
9,132539,30,78.4,81.0,52.0,74.0,2.8,13.0,1.0,144.0,0.8,58.0,0.5,15.0,205.0,26.0,33.5,80.0,4.4,1.9,78.0,0.0,1.5,41.0,63.33,108.0,137.0,39.0,110.0,221.0,16.0,97.0,117.0,37.9,3.1,0.24,90.0,11.2,7.39,54.0,170.2,0.0,4.0,0,2


## P1(f) Zero-padding

- Patients with **0 rows** in the window are excluded.
- The others are placed on the 13 window timestamps (12, 14, …, 36): bins with no data become all-zero rows,
  so row *i* always means hour 12 + 2*i*. Padded rows have `seg_entry_cnt = 0`.

In [12]:
STEPS = np.arange(12, 37, 2)  # the 13 window timestamps

def zero_pad(df):
    """Reindex each patient onto STEPS; missing bins become zero rows."""
    grid = pd.MultiIndex.from_product([df.pid.unique(), STEPS], names=["pid", "timestamp"])
    out = df.set_index(["pid", "timestamp"]).reindex(grid)
    out["target"] = out.groupby(level="pid").target.transform("first")
    return out.fillna(0).astype({"target": int, "seg_entry_cnt": int}).reset_index()

for name, piv, imp in [("Set A", pivot_a, imp_a), ("Set B", pivot_b, imp_b)]:
    excluded = sorted(set(piv.pid) - set(imp.pid))
    print(f"{name}: excluded {len(excluded)} patients with 0 rows in window: {excluded}")

pad_a, pad_b = zero_pad(imp_a), zero_pad(imp_b)
X_train = pad_a[FEATURE_COLS].to_numpy().reshape(-1, len(STEPS), len(FEATURE_COLS))
X_test = pad_b[FEATURE_COLS].to_numpy().reshape(-1, len(STEPS), len(FEATURE_COLS))
print("X_train:", X_train.shape, "| X_test:", X_test.shape)

Set A: excluded 8 patients with 0 rows in window: [133628, 138477, 139060, 140501, 140936, 141069, 141264, 141772]
Set B: excluded 8 patients with 0 rows in window: [142731, 142917, 142998, 143656, 145611, 147514, 150309, 150649]
X_train: (3992, 13, 41) | X_test: (3992, 13, 41)


In [13]:
PID_PAD = 132772  # only 7 of 13 bins observed
before = imp_a[imp_a.pid == PID_PAD]
print(f"Patient {PID_PAD} - before padding:", before.shape)
display(before)
after = pad_a[pad_a.pid == PID_PAD]
print("After padding:", after.shape)
display(after)

Patient 132772 - before padding: (7, 45)


Parameter,pid,timestamp,Weight,ALP,ALT,AST,Albumin,BUN,Bilirubin,Cholesterol,Creatinine,DiasABP,FiO2,GCS,Glucose,HCO3,HCT,HR,K,Lactate,MAP,MechVent,Mg,NIDiasABP,NIMAP,NISysABP,Na,PaCO2,PaO2,Platelets,RespRate,SaO2,SysABP,Temp,TroponinI,TroponinT,Urine,WBC,pH,Age,Height,Gender,ICUType,target,seg_entry_cnt
1166,132772,12,70.4,53.0,27.0,41.0,2.8,38.0,0.4,144.0,1.3,54.500000,1.00,11.0,152.0,29.0,33.6,124.5,3.8,4.9,71.50,1.0,2.3,83.5,101.650000,138.000000,142.0,58.0,145.0,236.0,19.0,97.0,107.500,35.8,3.1,0.38,72.0,11.2,7.29,68.0,170.2,1.0,3.0,0,1
1167,132772,16,70.4,53.0,27.0,41.0,2.8,38.0,0.4,144.0,1.3,54.500000,1.00,11.0,152.0,29.0,33.6,124.5,3.8,4.9,71.50,1.0,2.3,83.5,101.650000,138.000000,142.0,58.0,145.0,236.0,19.0,97.0,107.500,35.8,3.1,0.28,72.0,11.2,7.29,68.0,170.2,1.0,3.0,0,1
1168,132772,26,70.4,53.0,27.0,41.0,2.8,38.0,0.4,144.0,1.3,54.500000,1.00,11.0,152.0,29.0,33.6,124.5,3.8,4.9,71.50,1.0,2.3,83.5,101.650000,138.000000,142.0,58.0,145.0,236.0,19.0,97.0,107.500,35.8,3.1,0.30,72.0,11.2,7.29,68.0,170.2,1.0,3.0,0,1
1169,132772,30,70.4,53.0,27.0,41.0,2.8,38.0,0.4,144.0,1.3,54.500000,1.00,11.0,152.0,29.0,33.6,124.5,3.8,4.9,71.50,1.0,2.3,83.5,101.650000,138.000000,142.0,58.0,145.0,236.0,19.0,97.0,107.500,35.8,3.1,0.30,72.0,11.2,7.29,68.0,170.2,1.0,3.0,0,3
1170,132772,32,70.4,53.0,27.0,41.0,2.8,45.0,0.4,144.0,1.6,54.500000,1.00,11.0,188.0,27.0,38.7,100.0,4.7,4.9,71.50,1.0,2.5,55.0,71.092143,103.285714,146.0,58.0,145.0,287.0,19.0,97.0,107.500,35.8,3.1,0.53,72.0,16.1,7.29,68.0,170.2,1.0,3.0,0,16
1171,132772,34,70.4,53.0,27.0,41.0,2.8,45.0,0.4,144.0,1.6,55.500000,0.85,10.0,188.0,27.0,38.7,93.0,4.7,0.8,75.25,1.0,2.5,55.0,71.092143,103.285714,146.0,58.0,145.0,287.0,19.0,97.0,93.625,35.8,3.1,0.53,72.0,16.1,7.29,68.0,170.2,1.0,3.0,0,10
1172,132772,36,70.4,53.0,27.0,41.0,2.8,45.0,0.4,144.0,1.6,57.333333,0.60,7.0,188.0,27.0,38.7,88.0,4.7,0.8,73.00,1.0,2.5,55.0,71.092143,103.285714,146.0,52.0,96.0,287.0,19.0,97.0,100.000,36.8,3.1,0.53,72.0,16.1,7.31,68.0,170.2,1.0,3.0,0,5


After padding: (13, 45)


Parameter,pid,timestamp,Weight,ALP,ALT,AST,Albumin,BUN,Bilirubin,Cholesterol,Creatinine,DiasABP,FiO2,GCS,Glucose,HCO3,HCT,HR,K,Lactate,MAP,MechVent,Mg,NIDiasABP,NIMAP,NISysABP,Na,PaCO2,PaO2,Platelets,RespRate,SaO2,SysABP,Temp,TroponinI,TroponinT,Urine,WBC,pH,Age,Height,Gender,ICUType,target,seg_entry_cnt
1183,132772,12,70.4,53.0,27.0,41.0,2.8,38.0,0.4,144.0,1.3,54.500000,1.00,11.0,152.0,29.0,33.6,124.5,3.8,4.9,71.50,1.0,2.3,83.5,101.650000,138.000000,142.0,58.0,145.0,236.0,19.0,97.0,107.500,35.8,3.1,0.38,72.0,11.2,7.29,68.0,170.2,1.0,3.0,0,1
1184,132772,14,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0,0
1185,132772,16,70.4,53.0,27.0,41.0,2.8,38.0,0.4,144.0,1.3,54.500000,1.00,11.0,152.0,29.0,33.6,124.5,3.8,4.9,71.50,1.0,2.3,83.5,101.650000,138.000000,142.0,58.0,145.0,236.0,19.0,97.0,107.500,35.8,3.1,0.28,72.0,11.2,7.29,68.0,170.2,1.0,3.0,0,1
1186,132772,18,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0,0
1187,132772,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0,0
1188,132772,22,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0,0
1189,132772,24,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0,0
1190,132772,26,70.4,53.0,27.0,41.0,2.8,38.0,0.4,144.0,1.3,54.500000,1.00,11.0,152.0,29.0,33.6,124.5,3.8,4.9,71.50,1.0,2.3,83.5,101.650000,138.000000,142.0,58.0,145.0,236.0,19.0,97.0,107.500,35.8,3.1,0.30,72.0,11.2,7.29,68.0,170.2,1.0,3.0,0,1
1191,132772,28,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0,0.000,0.0,0.0,0.00,0.0,0.0,0.00,0.0,0.0,0.0,0.0,0,0
1192,132772,30,70.4,53.0,27.0,41.0,2.8,38.0,0.4,144.0,1.3,54.500000,1.00,11.0,152.0,29.0,33.6,124.5,3.8,4.9,71.50,1.0,2.3,83.5,101.650000,138.000000,142.0,58.0,145.0,236.0,19.0,97.0,107.500,35.8,3.1,0.30,72.0,11.2,7.29,68.0,170.2,1.0,3.0,0,3


## P1(g) Labels and class distribution

In [14]:
# One label per kept patient, in the same order as X
y_train = pad_a.groupby("pid", sort=False).target.first().to_numpy()
y_test = pad_b.groupby("pid", sort=False).target.first().to_numpy()
assert len(y_train) == len(X_train) and len(y_test) == len(X_test)

def class_dist(y):
    counts = pd.Series(y).value_counts().sort_index()
    return pd.DataFrame({"count": counts, "proportion": (counts / counts.sum()).round(4)})

pd.concat(
    {"y_train (Set A)": class_dist(y_train), "y_test (Set B)": class_dist(y_test)}, axis=1
).rename_axis("In-hospital_death")

y_train (Set A)            y_test (Set B)           
                            count proportion          count proportion
In-hospital_death                                                     
0                            3438     0.8612           3424     0.8577
1                             554     0.1388            568     0.1423

## P2 Baseline training

**Data preprocessing** — same patients, window (12–36 h), labels and sentinel cleaning as P1, with two changes:
- **No imputation / zero-padding.** LightGBM handles `NaN` natively, and *whether* a variable was measured is
  itself informative (labs are ordered when clinicians are worried); filled-in or zero values would hide this.
- **Per-patient summary features.** The 13 bins are summarised as `last`, `mean`, `min`, `max` and `count`
  of each of the 36 dynamic variables, plus the 5 static descriptors (185 features).
  Trees need fixed-length tabular input, and `count` keeps the measurement frequency.

**Method** — LightGBM (gradient-boosted trees): strong on small tabular clinical data (~4k patients),
insensitive to outliers and feature scale, and deterministic on CPU. Hyperparameters are fixed a priori
(shallow trees, small learning rate, subsampling) and checked by 5-fold stratified CV on **Set A only**;
Set B is used only for the final evaluation. Every preprocessing statistic is learned from the training part only:
Set A for the final model, the training folds inside CV (e.g. P1's imputation fallbacks are refit per fold).

**Reproduce** — run all P1 cells, then the P2 cells below (final seed `SEED = 42`).

In [15]:
import lightgbm as lgb
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
print("lightgbm", lgb.__version__)

lightgbm 4.7.0


In [16]:
DYNAMIC = [c for c in FEATURE_COLS if c not in STATIC]
STATS = ["last", "mean", "min", "max", "count"]

def summary_features(win, pids):
    """One row per patient: STATS of each dynamic variable over its window bins + static descriptors."""
    g = clean_sentinels(win).groupby("pid")
    parts = [g[DYNAMIC].agg(s).add_suffix(f"_{s}") for s in STATS] + [g[STATIC].first()]
    return pd.concat(parts, axis=1).loc[pids]

# Same patients and order as P1's X / y
F_train = summary_features(win_a, pad_a.pid.unique())
F_test = summary_features(win_b, pad_b.pid.unique())
print("F_train:", F_train.shape, "| F_test:", F_test.shape)

F_train: (3992, 185) | F_test: (3992, 185)


In [17]:
PARAMS = dict(
    n_estimators=500, learning_rate=0.02,     # many small boosting steps
    num_leaves=15, min_child_samples=20,      # shallow trees for ~4k patients
    subsample=0.8, subsample_freq=1, colsample_bytree=0.5, reg_lambda=1.0,  # regularization
    deterministic=True, force_col_wise=True, verbose=-1,                    # reproducible runs
)

def make_model(seed=SEED):
    return lgb.LGBMClassifier(**PARAMS, random_state=seed)

def scores(y, p):
    return {"AUROC": roc_auc_score(y, p), "AUPRC": average_precision_score(y, p)}

def cv_scores(build, y, seed=SEED, k=5):
    """Out-of-fold scores on Set A only. build(tr, va) -> (X_tr, X_va) must learn statistics from tr only."""
    oof = np.zeros(len(y))
    for tr, va in StratifiedKFold(k, shuffle=True, random_state=seed).split(np.zeros(len(y)), y):
        X_tr, X_va = build(tr, va)
        oof[va] = make_model(seed).fit(X_tr, y[tr]).predict_proba(X_va)[:, 1]
    return scores(y, oof)

In [18]:
pids_train = pad_a.pid.unique()  # same order as y_train

def p1_tensor(win, pids, fallback):
    """P1 (e)+(f) for the given patients -> flattened (n, 13*41) matrix."""
    padded = zero_pad(impute(win[win.pid.isin(pids)], fallback))
    assert (padded.pid.unique() == pids).all()
    return pd.DataFrame(padded[FEATURE_COLS].to_numpy().reshape(len(pids), -1))

def p1_split(tr, va):
    fallback_tr = fit_fallback(win_a[win_a.pid.isin(pids_train[tr])])  # training fold only
    return p1_tensor(win_a, pids_train[tr], fallback_tr), p1_tensor(win_a, pids_train[va], fallback_tr)

def summary_split(tr, va):
    return F_train.iloc[tr], F_train.iloc[va]  # per-patient statistics: nothing learned across patients

# Design check: P1 tensor as-is vs. summary features
candidates = {"P1 tensor as-is (13x41 flattened)": p1_split, "Summary features (final)": summary_split}
pd.DataFrame({name: cv_scores(build, y_train) for name, build in candidates.items()}).T.round(4)

,AUROC,AUPRC
P1 tensor as-is (13x41 flattened),0.8317,0.4522
Summary features (final),0.8378,0.4782


In [19]:
# Final model: train on all of Set A, evaluate once on Set B
model = make_model().fit(F_train, y_train)
test = scores(y_test, model.predict_proba(F_test)[:, 1])

TIERS = [("Strong", 0.83, 0.46), ("Medium", 0.80, 0.43), ("Simple", 0.75, 0.38)]
tier = next((name for name, roc, prc in TIERS if test["AUROC"] >= roc and test["AUPRC"] >= prc), "none")
print(f"Set B (seed {SEED}): AUROC {test['AUROC']:.4f} | AUPRC {test['AUPRC']:.4f} -> tier: {tier}")

Set B (seed 42): AUROC 0.8545 | AUPRC 0.5122 -> tier: Strong


In [20]:
# Seed robustness: every seed tried (42 is the final one)
SEEDS = [42, 7, 123, 256, 1001]
runs = pd.DataFrame(
    {s: scores(y_test, make_model(s).fit(F_train, y_train).predict_proba(F_test)[:, 1]) for s in SEEDS}
).T.rename_axis("seed")
pd.concat([runs, runs.agg(["mean", "std"])]).round(4)

,AUROC,AUPRC
42,0.8545,0.5122
7,0.8566,0.5202
123,0.8547,0.5097
256,0.8552,0.5152
1001,0.8533,0.5026
mean,0.8549,0.5120
std,0.0012,0.0065
